# Assignment 5: RAG Agent

**Goal:** Build a Retrieval-Augmented Generation (RAG) agent that:
1. **Loads** a blog post about Prompt Engineering (Lilian Weng)
2. **Splits & embeds** the content into a vector store
3. **Creates a ReAct agent** with a `retrieve` tool
4. The agent can **search the blog post** to answer questions accurately

**Data source:** Different from Assignment 4 - uses a web blog post instead of Vision 2030 PDF

## 1. Setup & Imports

In [4]:
import os
import bs4
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
)

print("Model initialized successfully")

Model initialized successfully


## 2. Load Blog Post from Web

In [5]:
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()
print(f"Loaded {len(docs)} document(s) from blog post")
print(f"Content length: {len(docs[0].page_content)} characters")

Loaded 1 document(s) from blog post
Content length: 29286 characters


## 3. Split & Embed

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"Split into {len(all_splits)} chunks")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Split into 43 chunks


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3941.75it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Store in Vector Store

In [7]:
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
print("Stored all chunks in vector store")

Stored all chunks in vector store


## 5. RAG Agent with Retrieve Tool

In [8]:
@tool
def retrieve(query: str) -> str:
    """Search the prompt engineering blog post for relevant information."""
    retrieved_docs = vector_store.similarity_search(query, k=3)
    return "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )

agent = create_react_agent(
    model=model,
    tools=[retrieve],
    prompt=(
        "You are a helpful assistant that answers questions about prompt engineering techniques. "
        "You have a retrieve tool that searches a blog post for relevant info. "
        "Use it to answer questions. You can call it multiple times with different queries if needed."
    ),
)

print("RAG agent created with retrieve tool")

RAG agent created with retrieve tool


C:\Users\Yaz00\AppData\Local\Temp\ipykernel_4704\978059594.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 6. Demo: Simple Question

In [9]:
print("=" * 70)
print("Question: What is few-shot prompting?")
print("=" * 70)

result = agent.invoke({
    "messages": [HumanMessage(content="What is few-shot prompting?")]
})
print(f"\nAnswer: {result['messages'][-1].content}")

Question: What is few-shot prompting?

Answer: [{'type': 'text', 'text': 'Few-shot prompting is a technique where a few examples, or "demonstrations," are included in the prompt to help explain your intent and describe the task instructions to the model. This method allows the model to better understand the user\'s intention and follow instructions.\n\nHowever, few-shot prompting can be costly due to token usage and can limit the input length because of context length restrictions. A related concept is "in-context instruction learning," which combines few-shot learning with instruction prompting by incorporating multiple demonstration examples for different tasks within the prompt.', 'extras': {'signature': 'CrAEAb4+9vv36Eda167diHinnTxEsQ40U8NM9//htJ1ReJxVfl5rhjq/OovJLNbCxF+D43cnfcZZjdrpQODuKyTEgNZp9+x/+/+y/UVqDDDE/jh5Tm2TdmQSRY7dhROeeKSNp/j/LebuKWmGqYgGU9sd5jexefPyJOPrJeKjjpOZHy9VfvPjBEPAoB2yjtKgHmU4qaXkmkAQZvdrJSFqeCd5yrMWUKhVte51jeCFaijyJZ2zNDfRuH79hSJ/6qNFFvqLDxGsGaTwedDTnzlreFX/y6

## 7. Demo: Multi-step Question

In [10]:
print("=" * 70)
print("Question: What is Chain of Thought prompting? And how does it compare to zero-shot prompting?")
print("=" * 70)

result = agent.invoke({
    "messages": [HumanMessage(
        content="What is Chain of Thought prompting? And how does it compare to zero-shot prompting?"
    )]
})
print(f"\nAnswer: {result['messages'][-1].content}")

Question: What is Chain of Thought prompting? And how does it compare to zero-shot prompting?

Answer: [{'type': 'text', 'text': 'Chain of Thought (CoT) prompting is a technique where a model generates a sequence of short sentences that describe its step-by-step reasoning process, leading to a final answer. This method is particularly effective for complex reasoning tasks and with larger language models (e.g., those with over 50 billion parameters), where it significantly enhances performance. For simpler tasks, the benefits are less pronounced.\n\nZero-shot prompting, on the other hand, involves providing direct instructions to a model without any examples. The model is expected to understand the intent from the instruction alone and generate a response based on its pre-trained knowledge.\n\nHere\'s how they compare:\n\n*   **Provision of Examples:** Standard zero-shot prompting does not include any examples in the prompt, relying solely on the instruction. CoT prompting often involve